# Campaign Text Analytics using NLP
In a digital marketing setting, companies run multiple campaigns with varying ad copies (texts). Some ad messages perform well (high CTR or conversions), while others underperform.

Goal: Use Natural Language Processing (NLP) to analyze the textual content of ads and uncover:

Common themes or clusters of messaging

Sentiment or tone used in each ad

How textual features relate to performance metrics like CTR

Actionable insights for marketers to write better-performing ad copy


What Are We Trying to Discover?
What kinds of ad messages exist?

E.g., “discount-focused,” “feature-focused,” “emotional appeal”

We’ll use embedding + clustering for this

Does message type correlate with performance (CTR, conversions)?

E.g., Do “emotional tone” ads perform better?

We’ll use sentiment analysis + basic regression/visuals for this

Can we build a dashboard to make this accessible to marketers?

Yes — using Streamlit at the end

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
import re
from wordcloud import WordCloud

In [5]:
import sys
print(sys.version)

3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]


In [6]:
# Loading spacy model

nlp = spacy.load("en_core_web_sm")

In [7]:
df = pd.read_csv("../Data/mock_ad_campaign_data.csv")

In [8]:
df

,ad_text,campaign_name,budget,clicks,impressions,CTR
0,Get 50% off on all shoes this weekend only!,Footwear Sale,5000,250,5000,0.050000
1,Try our new organic face cream for glowing skin.,Skincare Launch,3000,180,4500,0.040000
2,Upgrade to premium and enjoy ad-free streaming.,Subscription Promo,7000,300,10000,0.030000
3,Delicious pizza delivered hot and fresh to you...,Food Delivery,4500,270,7000,0.038571
4,Limited time offer on mobile phones with free ...,Mobile Offers,6000,320,9500,0.033684
5,Join our gym today and get your first month free.,Fitness Deal,3500,200,6000,0.033333
6,Travel to Europe with 20% off on early bookings!,Travel Promo,8000,330,11000,0.030000
7,Don't miss out – big discounts on winter jackets.,Winter Sale,4000,290,8000,0.036250
8,Healthy snacks for your busy lifestyle – shop ...,Healthy Living,2500,160,5000,0.032000
9,Learn Python online with expert instructors – ...,Online Learning,5500,310,9000,0.034444


In [9]:
test = nlp(df.iloc[7,0])

In [10]:
for token in test:
    print(token.text, token.lemma_, token.pos_, token.is_stop)

Do do AUX True
n't not PART True
miss miss VERB False
out out ADP True
– – PUNCT False
big big ADJ False
discounts discount NOUN False
on on ADP True
winter winter NOUN False
jackets jacket NOUN False
. . PUNCT False


### Preprocessing and lemmatizing the text

In [11]:
# Function to perform basic text cleaning
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]','',text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.is_alpha]
    return " ".join(tokens)

In [12]:
# Create a column with cleaned text
df['cleaned_text'] = df['ad_text'].apply(preprocess)

# Preview
print(df[['ad_text', 'cleaned_text']].head())

                                             ad_text  \
0        Get 50% off on all shoes this weekend only!   
1   Try our new organic face cream for glowing skin.   
2    Upgrade to premium and enjoy ad-free streaming.   
3  Delicious pizza delivered hot and fresh to you...   
4  Limited time offer on mobile phones with free ...   

                                    cleaned_text  
0                                   shoe weekend  
1           try new organic face cream glow skin  
2         upgrade premium enjoy adfree streaming  
3         delicious pizza deliver hot fresh door  
4  limited time offer mobile phone free delivery  


In [13]:
df['cleaned_text']

0                                         shoe weekend
1                 try new organic face cream glow skin
2               upgrade premium enjoy adfree streaming
3               delicious pizza deliver hot fresh door
4        limited time offer mobile phone free delivery
5                            join gym today month free
6                          travel europe early booking
7                  not miss big discount winter jacket
8                    healthy snack busy lifestyle shop
9    learn python online expert instructor enroll t...
Name: cleaned_text, dtype: object

In [14]:
# Visualize the words
all_text = " ".join(df['cleaned_text'])

wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='tab10').generate(all_text)

plt.figure(figsize=(12,8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Word Cloud of Campaign Ad Texts", fontsize=16)
plt.show()


C:\Users\manu.krishnan\AppData\Local\Temp\ipykernel_17532\1038276876.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Cluster the similar ad texts into themes

#### Using TF-IDF, n-grams & KMeans clustering

In [15]:
# Perform TF-IDF vectorization using n-gram
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=1000,  max_df=0.9, min_df=2)
X_tfidf = vectorizer.fit_transform(df['cleaned_text'])
print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (10, 2)


In [16]:
from sklearn.cluster import KMeans

# Choose number of clusters
num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
clusters = kmeans.fit_predict(X_tfidf)

df['ad_theme_clusters'] = clusters

In [17]:
for i in range(num_clusters):
    print(f"\n=== Theme{i} ====")
    print(df[(df['ad_theme_clusters']==i)]['ad_text'].values)


=== Theme0 ====
['Get 50% off on all shoes this weekend only!'
 'Try our new organic face cream for glowing skin.'
 'Upgrade to premium and enjoy ad-free streaming.'
 'Delicious pizza delivered hot and fresh to your door.'
 'Travel to Europe with 20% off on early bookings!'
 "Don't miss out – big discounts on winter jackets."
 'Healthy snacks for your busy lifestyle – shop now.']

=== Theme1 ====
['Join our gym today and get your first month free.']

=== Theme2 ====
['Limited time offer on mobile phones with free delivery.']

=== Theme3 ====
['Learn Python online with expert instructors – enroll today.']


#### Perform word vectorization & clustering using Word2vec & Kmeans

In [18]:
from gensim.models import Word2Vec

In [19]:
df['tokens'] = df['cleaned_text'].apply(lambda x: x.split())  # Convert sentences into tokens

In [20]:
df['tokens'].tolist()

[['shoe', 'weekend'],
 ['try', 'new', 'organic', 'face', 'cream', 'glow', 'skin'],
 ['upgrade', 'premium', 'enjoy', 'adfree', 'streaming'],
 ['delicious', 'pizza', 'deliver', 'hot', 'fresh', 'door'],
 ['limited', 'time', 'offer', 'mobile', 'phone', 'free', 'delivery'],
 ['join', 'gym', 'today', 'month', 'free'],
 ['travel', 'europe', 'early', 'booking'],
 ['not', 'miss', 'big', 'discount', 'winter', 'jacket'],
 ['healthy', 'snack', 'busy', 'lifestyle', 'shop'],
 ['learn', 'python', 'online', 'expert', 'instructor', 'enroll', 'today']]

In [21]:
# Train the Word2Vec Model
def train_word2vec(tokenized_texts, vector_size=100, window=5, min_count=1):
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4
    )
    return model

In [22]:
word2vec_model = train_word2vec(df['tokens'].tolist())

In [23]:
word2vec_model.wv.most_similar('travel')

[('door', 0.24816644191741943),
 ('healthy', 0.2287854105234146),
 ('skin', 0.18983878195285797),
 ('europe', 0.17163145542144775),
 ('hot', 0.16691496968269348),
 ('enroll', 0.11916332691907883),
 ('phone', 0.11493445187807083),
 ('expert', 0.10897885262966156),
 ('big', 0.10787872225046158),
 ('miss', 0.08625233173370361)]

In [24]:
df['cleaned_text']

0                                         shoe weekend
1                 try new organic face cream glow skin
2               upgrade premium enjoy adfree streaming
3               delicious pizza deliver hot fresh door
4        limited time offer mobile phone free delivery
5                            join gym today month free
6                          travel europe early booking
7                  not miss big discount winter jacket
8                    healthy snack busy lifestyle shop
9    learn python online expert instructor enroll t...
Name: cleaned_text, dtype: object

In [25]:
# Create Document Embeddings
def get_doc_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)

In [26]:
# Apply to the Dataframe
df['doc_vector'] = df['tokens'].apply(lambda tokens: get_doc_vector(tokens, word2vec_model))

# Stack vectors into a matrix
doc_vectors = np.vstack(df['doc_vector'].values)

In [27]:
df['doc_vector']

0    [0.002480599, 0.0005589329, 0.004202884, -0.00...
1    [0.0026168749, 0.0012093342, 0.0021884777, 0.0...
2    [-0.0019276931, 0.0015687865, 0.0019126035, -0...
3    [0.0030699323, -6.483247e-06, 0.0022564428, 0....
4    [-0.0028134717, 0.0027127385, -0.0019288564, -...
5    [-0.0009916786, -0.00020802219, 0.0010198652, ...
6    [-0.0022781307, 0.0032137367, -0.0016086414, -...
7    [-0.0008623141, -0.00072353333, -0.006819255, ...
8    [0.0037670534, -0.0023665274, -0.00086013286, ...
9    [-0.0019055778, -0.0026972329, -0.0005447854, ...
Name: doc_vector, dtype: object

In [28]:
# Dimensionality reduction using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca_vectors = pca.fit_transform(doc_vectors)

In [29]:
#KMeans Clustering
kmeans = KMeans(n_clusters=4, random_state=42)
df['w2v_cluster'] = kmeans.fit_predict(pca_vectors)

In [30]:
pca_vectors

array([[-0.03967365, -0.0019087 ],
       [ 0.00264654,  0.00613546],
       [ 0.00829537,  0.01878312],
       [ 0.00288572, -0.00490962],
       [ 0.00516496,  0.00125728],
       [ 0.00198525,  0.0059733 ],
       [ 0.0162507 , -0.01686184],
       [ 0.00096265, -0.00163043],
       [-0.00087022, -0.01232262],
       [ 0.00235267,  0.00548405]], dtype=float32)

In [31]:
df['w2v_cluster']

0    2
1    1
2    1
3    0
4    1
5    1
6    3
7    0
8    0
9    1
Name: w2v_cluster, dtype: int32

In [32]:
# Visualize the clusters
plt.figure(figsize=(10,7))
colors = ['red', 'blue', 'green', 'purple']
for i in range(4):
    plt.scatter(pca_vectors[df['w2v_cluster'] == i,0],
                pca_vectors[df['w2v_cluster'] == i,1],
                label=f"Cluster {i}", color=colors[i], alpha=0.6)

plt.title("Ad Text Clusters via Word2Vec + KMeans")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.show()        

C:\Users\manu.krishnan\AppData\Local\Temp\ipykernel_17532\883012528.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
for i in range(4):
    print(f"\nCluster {i} Examples:")
    print(df[df['w2v_cluster'] == i]['ad_text'].head(3).to_string(index=False))


Cluster 0 Examples:
Delicious pizza delivered hot and fresh to your...
 Don't miss out – big discounts on winter jackets.
Healthy snacks for your busy lifestyle – shop now.

Cluster 1 Examples:
  Try our new organic face cream for glowing skin.
   Upgrade to premium and enjoy ad-free streaming.
Limited time offer on mobile phones with free d...

Cluster 2 Examples:
Get 50% off on all shoes this weekend only!

Cluster 3 Examples:
Travel to Europe with 20% off on early bookings!


#### Perform vectorization using sentence-BERT

In [34]:
from sentence_transformers import SentenceTransformer

# Load pre-trained Sentence-BERT model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight & fast


C:\Users\manu.krishnan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
sbert_embeddings = sbert_model.encode(df['cleaned_text'].tolist(), show_progress_bar = True)

# Convert to NumPy array for clustering
sbert_embeddings = np.array(sbert_embeddings)

Batches: 100%|███████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  6.33it/s]


In [36]:
# Reduce the dimensions using PCA or t-SNE

# --- PCA (faster, linear) ---
pca = PCA(n_components=2, random_state=42)
bert_pca = pca.fit_transform(sbert_embeddings)

In [37]:
# Visualize the embeddings

plt.figure(figsize=(10,7))
plt.scatter(bert_pca[:,0], bert_pca[:,1], color = 'blue', alpha=0.7)

for i, text in enumerate(df['cleaned_text']):
    plt.text(bert_pca[i, 0] + 0.3, bert_pca[i, 1], text[:20], fontsize=8)

plt.title("Sentence-BERT Embeddings Visualized with t-SNE")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.grid(True)
plt.tight_layout()
plt.show()

C:\Users\manu.krishnan\AppData\Local\Temp\ipykernel_17532\608241486.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [38]:
# Perform KMeans Clustering
num_clusters = 3
kmeans_bert = KMeans(n_clusters=3, random_state=42)
df['cluster_bert'] = kmeans_bert.fit_predict(sbert_embeddings)

In [39]:
# Evaluate clustering quality using Silhouette score. A higher score (closer to 1) means better-defined clusters.
from sklearn.metrics import silhouette_score
sil_score = silhouette_score(sbert_embeddings, df['cluster_bert'])
print(f"silhouette score: {sil_score: .2f}")

silhouette score:  0.03


In [40]:
for i in range(num_clusters):
    print(f"\n----- cluster {i} -----")
    print(df[df['cluster_bert'] == i]['ad_text'].values)


----- cluster 0 -----
['Delicious pizza delivered hot and fresh to your door.'
 'Limited time offer on mobile phones with free delivery.'
 'Travel to Europe with 20% off on early bookings!'
 'Healthy snacks for your busy lifestyle – shop now.']

----- cluster 1 -----
['Try our new organic face cream for glowing skin.'
 'Upgrade to premium and enjoy ad-free streaming.'
 "Don't miss out – big discounts on winter jackets."]

----- cluster 2 -----
['Get 50% off on all shoes this weekend only!'
 'Join our gym today and get your first month free.'
 'Learn Python online with expert instructors – enroll today.']


### Sentiment & KPI Analysis

In [41]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

In [42]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\manu.krishnan\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [43]:
# Initialize the sentiment analyzer
sia = SentimentIntensityAnalyzer()

In [44]:
# Aplly sentiment scoring
df['sentiment'] = df['cleaned_text'].apply(lambda x: sia.polarity_scores(x)['compound'])

In [45]:
# Compare average CTR per sentiment score
plt.scatter(df['sentiment'], df['CTR'])
plt.xlabel("Sentiment Score")
plt.ylabel("CTR")
plt.title("Sentiment vs CTR")
plt.grid(True)
plt.show()

C:\Users\manu.krishnan\AppData\Local\Temp\ipykernel_17532\3725231845.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [46]:
# Box plot of CTR by cluster
import seaborn as sns
sns.boxplot(x='cluster_bert', y='CTR', data=df)
plt.title("CTR Distribution by Cluster")
plt.show()

C:\Users\manu.krishnan\AppData\Local\Temp\ipykernel_17532\1670566602.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### ANOVA to test significance
ANOVA (Analysis of Variance) is used to test whether the means of multiple groups are significantly different from each other.
Each cluster groups similar Ad messages. If ads in different clusters have significantly different CTRs, that tells us,
- Certain Ad styles/themes perform better.
- We can optimize Ad writing based on this insight.

In [47]:
from statsmodels.formula.api import ols
import statsmodels.api as sm

In [48]:
# Fit an OLS model : CTR explained by clusters(as categorical variable)
anova_model = ols('CTR ~ C(cluster_bert)', data=df).fit()

In [49]:
# Check  ANOVA
anova_table = sm.stats.anova_lm(anova_model, typ=2)
print(anova_table)

                   sum_sq   df         F    PR(>F)
C(cluster_bert)  0.000056  2.0  0.744181  0.509283
Residual         0.000265  7.0       NaN       NaN


In [50]:
df.to_csv("campaign_text_analytics_modelled_output.csv", index=False)

### Streamlit App Code

In [1]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn as sns
import io

st.set_page_config(page_title="Campaign Text Analytics Dashboard", layout="wide")
st.title("📊 Campaign Text Analytics Dashboard")

st.markdown("""
This app allows you to:
- Upload processed ad campaign data
- Explore clusters, sentiment, and CTR
- Visualize insights for optimization
""")

# Upload CSV
    uploaded_file = st.file_uploader("Upload your CSV file", type=["csv"])

if uploaded_file is not None:
    df = pd.read_csv(uploaded_file)

    # Validate necessary columns
    required_cols = {'cleaned_text', 'cluster', 'sentiment', 'CTR'}
    if not required_cols.issubset(df.columns):
        st.error(f"CSV must contain these columns: {required_cols}")
    else:
        # Show sample data
        st.subheader("🔍 Preview of Uploaded Data")
        st.dataframe(df.head())

        # KPI Overview
        st.subheader("📈 Key Metrics by Cluster")
        kpi_summary = df.groupby('cluster').agg(
            avg_ctr=('CTR', 'mean'),
            count=('CTR', 'count'),
            avg_sentiment=('sentiment', 'mean')
        ).reset_index()
        st.dataframe(kpi_summary)

        # Visualization options
        st.subheader("📊 Visual Analysis")
        viz_option = st.selectbox("Choose visualization", [
            "CTR by Cluster", "Sentiment vs CTR", "CTR Distribution", "Sentiment Distribution"
        ])

        if viz_option == "CTR by Cluster":
            fig, ax = plt.subplots()
            sns.boxplot(data=df, x='cluster', y='CTR', ax=ax)
            ax.set_title("CTR Distribution per Cluster")
            st.pyplot(fig)

        elif viz_option == "Sentiment vs CTR":
            fig, ax = plt.subplots()
            sns.scatterplot(data=df, x='sentiment', y='CTR', hue='cluster', palette='tab10', ax=ax)
            ax.set_title("Sentiment vs CTR by Cluster")
            st.pyplot(fig)

        elif viz_option == "CTR Distribution":
            fig, ax = plt.subplots()
            sns.histplot(df['CTR'], bins=20, kde=True, ax=ax)
            ax.set_title("Overall CTR Distribution")
            st.pyplot(fig)

        elif viz_option == "Sentiment Distribution":
            fig, ax = plt.subplots()
            sns.histplot(df['sentiment'], bins=20, kde=True, ax=ax)
            ax.set_title("Sentiment Score Distribution")
            st.pyplot(fig)

        # Option to download
        st.subheader("⬇️ Download Processed Data")
        csv_download = df.to_csv(index=False).encode('utf-8')
        st.download_button("Download CSV", csv_download, "processed_campaign_data.csv", "text/csv")

else:
    st.info("Please upload a CSV file to begin.")


2025-05-01 04:51:58.243 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 04:51:58.259 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 04:51:59.978 
  command:

    streamlit run C:\Users\manu.krishnan\AppData\Local\Programs\Python\Python311\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-05-01 04:51:59.978 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 04:51:59.978 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 04:51:59.978 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-01 04:51:59.994 Thread 'MainThread': missing ScriptRunContext!